# Track C Colab Runner (Streaming)

Runs Track C end-to-end in Colab **without uploading raw WikiArt parquet** by streaming from Hugging Face.

## 0) Runtime

- Runtime -> Change runtime type -> GPU
- High-RAM runtime recommended

In [ ]:
# 0.5) Create Track C workspace in /content and show upload checklist
from pathlib import Path
import os

WORK_DIR = Path('/content/trackc_workspace')
folders = [
    WORK_DIR,
    WORK_DIR / 'scripts',
    WORK_DIR / 'artifacts',
    WORK_DIR / 'artifacts' / 'runs',
    WORK_DIR / 'artifacts' / 'linear_map',
    WORK_DIR / 'configs',
    WORK_DIR / 'data',
    WORK_DIR / 'data' / 'labeled',
    WORK_DIR / 'logs',
    WORK_DIR / 'notebooks',
]
for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)

placeholder_files = [
    WORK_DIR / 'scripts' / 'train_linear_map.py',
    WORK_DIR / 'scripts' / 'benchmark_linear_map.py',
    WORK_DIR / 'scripts' / 'run_track_c_comparison.py',
    WORK_DIR / 'scripts' / 'merge_track_a_runs.py',
    WORK_DIR / 'requirements.txt',
]
for file_path in placeholder_files:
    file_path.touch(exist_ok=True)

os.chdir(WORK_DIR)
print('Workspace ready:', WORK_DIR)
print('CWD:', Path.cwd())

print('\nUpload/replace these files in place:')
print('  scripts/train_linear_map.py')
print('  scripts/benchmark_linear_map.py')
print('  scripts/run_track_c_comparison.py')
print('  scripts/merge_track_a_runs.py')
print('  requirements.txt')

print('\nOptional folders to upload for final A+B+C comparison in this runtime:')
print('  artifacts/runs/trackA_*  (or at least artifacts/runs/trackA_merged)')
print('  artifacts/runs/trackB_*')

print('\nRaw WikiArt parquet upload is NOT required (streaming mode is used).')

In [ ]:
# 1) Mount Google Drive (for saving outputs)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2) Install dependencies
%pip install -q -r requirements.txt sentencepiece protobuf

In [ ]:
# 3) Optional: set HF token for more reliable/faster streaming
import os
from getpass import getpass
tok = getpass('HF_TOKEN (optional): ').strip()
if tok:
    os.environ['HF_TOKEN'] = tok
    print('HF_TOKEN set for this runtime session')
else:
    print('No HF token set; continuing unauthenticated')

In [ ]:
# 4) Preflight check for required files
from pathlib import Path
required = [
    'scripts/train_linear_map.py',
    'scripts/benchmark_linear_map.py',
    'scripts/run_track_c_comparison.py',
    'scripts/merge_track_a_runs.py',
]
missing = [p for p in required if not Path(p).exists()]
print('Missing required files:', missing if missing else 'None')
if missing:
    raise FileNotFoundError('Upload missing files before running Track C.')

In [ ]:
# 5) Configure run parameters
STREAMING_REPO = 'huggan/wikiart'
STREAMING_SPLIT = 'train'
TRAIN_MAX_ROWS = 11320        # matches your Track B scale
EVAL_MAX_IMAGES = 4752        # practical full-eval target; set 0 for full stream
TEMPLATES_PER_CLASS = 1
RUN_ID = ''                   # optional: e.g. 'trackC_full_20260315'

print('STREAMING_REPO:', STREAMING_REPO)
print('TRAIN_MAX_ROWS:', TRAIN_MAX_ROWS)
print('EVAL_MAX_IMAGES:', EVAL_MAX_IMAGES)

In [ ]:
# 6) Train production linear map W from streamed data
train_cmd = (
    f"python scripts/train_linear_map.py "
    f"--streaming-repo-id {STREAMING_REPO} "
    f"--streaming-split {STREAMING_SPLIT} "
    f"--max-rows {TRAIN_MAX_ROWS} "
    f"--style-template 'a painting in style_{{style_id}}' "
    f"--output-dir artifacts/linear_map "
    f"--log-every 500"
 )
print(train_cmd)
!{train_cmd}

# quick existence check
from pathlib import Path
print('W exists:', Path('artifacts/linear_map/linear_map_W.npy').exists())
print('meta exists:', Path('artifacts/linear_map/linear_map_meta.json').exists())

In [ ]:
# 7) Smoke run (quick validation)
!python scripts/run_track_c_comparison.py \
  --streaming-repo-id {STREAMING_REPO} \
  --streaming-split {STREAMING_SPLIT} \
  --linear-map-path artifacts/linear_map/linear_map_W.npy \
  --output-root artifacts/runs \
  --targets style genre \
  --templates-per-class 1 \
  --max-images 64 \
  --run-id trackC_smoke_colab

In [ ]:
# 8) Full Track C run (streaming)
run_id_arg = f'--run-id {RUN_ID}' if RUN_ID.strip() else ''
cmd = f"python scripts/run_track_c_comparison.py --streaming-repo-id {STREAMING_REPO} --streaming-split {STREAMING_SPLIT} --linear-map-path artifacts/linear_map/linear_map_W.npy --output-root artifacts/runs --targets style genre --templates-per-class {TEMPLATES_PER_CLASS} --max-images {EVAL_MAX_IMAGES} {run_id_arg}"
print(cmd)
!{cmd}

In [ ]:
# 9) Optional: merge Track A + B + C (if those runs exist in this runtime)
from pathlib import Path

have_a = any(Path('artifacts/runs').glob('trackA_*'))
have_b = any(Path('artifacts/runs').glob('trackB_*'))
have_c = any(Path('artifacts/runs').glob('trackC_*'))
print('Track A present:', have_a, '| Track B present:', have_b, '| Track C present:', have_c)

if have_c and (have_a or have_b):
    get_ipython().system('python scripts/merge_track_a_runs.py --runs-root artifacts/runs --run-globs trackA_* trackB_* trackC_* --output-dir artifacts/runs/trackABC_merged')
else:
    print('Skipping A+B+C merge: need trackC_* and at least one of trackA_* or trackB_* in artifacts/runs')

In [ ]:
# 10) Save outputs to Drive
from pathlib import Path
import shutil

DRIVE_OUT = Path('/content/drive/MyDrive/clip_categorizer_outputs')
DRIVE_OUT.mkdir(parents=True, exist_ok=True)

# Copy Track C runs
runs_root = Path('artifacts/runs')
for run_dir in runs_root.glob('trackC_*'):
    dest = DRIVE_OUT / 'runs' / run_dir.name
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists():
        shutil.rmtree(dest)
    shutil.copytree(run_dir, dest)

# Copy linear map artifacts
linear_map_src = Path('artifacts/linear_map')
linear_map_dst = DRIVE_OUT / 'linear_map'
if linear_map_src.exists():
    if linear_map_dst.exists():
        shutil.rmtree(linear_map_dst)
    shutil.copytree(linear_map_src, linear_map_dst)

# Copy merged A+B+C table if present
merge_src = Path('artifacts/runs/trackABC_merged')
if merge_src.exists():
    merge_dst = DRIVE_OUT / 'trackABC_merged'
    if merge_dst.exists():
        shutil.rmtree(merge_dst)
    shutil.copytree(merge_src, merge_dst)

print('Saved outputs to:', DRIVE_OUT)

In [ ]:
# 11) Optional: zip outputs for download + auto-download to laptop
from pathlib import Path
from google.colab import files

zip_path = Path('artifacts/trackC_outputs.zip')
if zip_path.exists():
    zip_path.unlink()

!cd artifacts && zip -r trackC_outputs.zip runs/trackC_* linear_map runs/trackABC_merged || true

if zip_path.exists():
    print('Downloading:', zip_path)
    files.download(str(zip_path))
else:
    print('Zip file was not created; check artifacts/runs/trackC_* and artifacts/linear_map')

In [ ]:
# 12) Show key output locations
from pathlib import Path
print('Track C runs:')
for p in sorted(Path('artifacts/runs').glob('trackC_*')):
    print(' -', p)

summary_files = sorted(Path('artifacts/runs').glob('trackC_*/summary/track_c_summary.json'))
print('\nTrack C summaries:')
for p in summary_files:
    print(' -', p)

merged = Path('artifacts/runs/trackABC_merged/final_slides_table.csv')
if merged.exists():
    print('\nA+B+C merged table:', merged)
else:
    print('\nNo A+B+C merged table found yet (cell 9 will create it when A/B/C runs are present).')